# Imports and Setup

In [1]:
!pip show spicelib
# !pip install spicelib # install if needed
# !pip install --upgrade ipykernel jedi


Name: spicelib
Version: 1.4.5
Summary: A set of tools to Automate Spice simulations
Home-page: https://github.com/nunobrum/spicelib
Author: Nuno Brum
Author-email: nuno.brum@gmail.com
License: GPL-3.0
Location: /headless/.local/lib/python3.12/site-packages
Requires: matplotlib, numpy, psutil, scipy
Required-by: 


In [2]:
from spicelib import SpiceEditor, SimRunner, RawRead
from spicelib.simulators import ngspice_simulator 

import logging
import os
import shutil


from pathlib import Path


In [3]:
# --- Your notebook logger (unchanged) ---
# Create a logger
logger = logging.getLogger("notebook_logger")
logger.setLevel(logging.DEBUG)  # Set lowest level you want to capture (DEBUG, INFO, WARNING...)

# Create a console handler (for notebook output)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)  # You can choose INFO if DEBUG is too noisy
logger.propagate = False  # stop passing logs to root logger

# Create a formatter
formatter = logging.Formatter(
    fmt="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
console_handler.setFormatter(formatter)

# Add the handler to the logger (avoid duplicates)
if not logger.handlers:
    logger.addHandler(console_handler)

logger.info("notebook_logger initialized.")

# --- NEW: Enable debug logging for spicelib ---
spicelib_logger = logging.getLogger("spicelib")
spicelib_logger.setLevel(logging.DEBUG)

# Optionally, attach the same console handler so spicelib logs show up too
if not spicelib_logger.handlers:
    spicelib_logger.addHandler(console_handler)

logger.info("spicelib logger set to DEBUG")

07:03:08 [INFO] notebook_logger initialized.
07:03:08 [INFO] spicelib logger set to DEBUG


## Simulation Config

In [4]:
PATH_TO_NGSPICE = Path("/foss/tools/bin/ngspice")

PROJECT_NAME    = "tia-bpf-1"
SCHEMATIC_NAME  = "tb_ac"
SIZER_NAME      = "simple"

OUTPUT_DIR      = Path(f"./runs/{SIZER_NAME}/{PROJECT_NAME}")
INITIAL_NETLIST = Path(f"../../{PROJECT_NAME}/netlist/{SCHEMATIC_NAME}.spice")


if os.path.exists(OUTPUT_DIR):
    logger.warning(f"Output directory already exists, removing: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=False)

if not INITIAL_NETLIST.exists():
    raise FileNotFoundError(f"Initial netlist not found: {INITIAL_NETLIST}")


logger.info(f"Using ngspice from {PATH_TO_NGSPICE}")
logger.info(f"project: {PROJECT_NAME}, schematic: {SCHEMATIC_NAME}")

07:03:08 [WARNING] Output directory already exists, removing: runs/simple/tia-bpf-1
07:03:08 [INFO] Using ngspice from /foss/tools/bin/ngspice
07:03:08 [INFO] project: tia-bpf-1, schematic: tb_ac


# SpiceLib

## Create a runner

In [5]:
simulator = ngspice_simulator.NGspiceSimulator.create_from(path_to_exe=PATH_TO_NGSPICE)
simulator.set_compatibility_mode('lt') # this has to be "a" for complete netlist transform

In [6]:
simulator = ngspice_simulator.NGspiceSimulator.create_from(path_to_exe=PATH_TO_NGSPICE)
simulator.set_compatibility_mode("a")

runner = SimRunner(
    simulator=simulator, 
    output_folder=OUTPUT_DIR,
    cwd=OUTPUT_DIR,
    )

# runner.cwd = Path("./")

07:03:08 [INFO] SimRunner initialized


## Load the initial (unsized) TB schematic

In [7]:
# Create a SpiceEditor Instance
editor = SpiceEditor(netlist_file=INITIAL_NETLIST)

# Nodes
nodes = editor.get_all_nodes()
logger.info(f"Nodes in the netlist:\n{nodes}")

# Parameters
params = editor.get_all_parameter_names()
tb_params  = [(param, editor.get_parameter(param)) for param in params if not "X_DUT" in param]
dut_params = [(param, editor.get_parameter(param)) for param in params if "X_DUT" in param]
logger.info(f"Testbench parameters:\n{tb_params}")
logger.info(f"DUT parameters:\n{dut_params}")



07:03:08 [INFO] Nodes in the netlist:
['VSS', 'GND', 'VDD', 'Vop', 'Von', 'In', 'Ip', 'Vbias']
07:03:08 [INFO] Testbench parameters:
[('CLOAD', '1p'), ('ICM', '0.5e-6'), ('IDD', '1.8e-6'), ('VBIAS', '1.0'), ('VDD', '6.0')]
07:03:08 [INFO] DUT parameters:
[('X_DUT_CDD', '1n'), ('X_DUT_LDD', '2n'), ('X_DUT_M1_2_L', '0.40u'), ('X_DUT_M1_2_NF', '2'), ('X_DUT_M1_2_W', '0.22u'), ('X_DUT_R4', '1k'), ('X_DUT_RS', '1k')]


### Run a sanity
Make sure the schematic works

In [12]:
runtask = runner.run_now(
    netlist=INITIAL_NETLIST,
    exe_log=True )
runtask

07:09:44 [INFO] RunTask #5:: Starting simulation 5: runs/simple/tia-bpf-1/tb_ac_5.spice
07:09:52 [DEBUG] Running command: ['/foss/tools/bin/ngspice', '-D', 'ngbehavior=a', '-b', '-o', '/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_5.log', '-r', '/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_5.raw', '/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_5.spice'], with timeout: 600.0
07:13:11 [ERROR] RunTask #5:Simulation Aborted. Time elapsed: 03:26.0684


(None, PosixPath('runs/simple/tia-bpf-1/tb_ac_5.fail'))

In [ ]:
# editor.prepare_for_simulator(ngspice_simulator.Simulator)
# simulator._compatibility_mode

/foss/tools/bin/ngspice -D ngbehavior=a -b -o /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_2.log -r /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_2.raw /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/sizing/runs/simple/tia-bpf-1/tb_ac_2.spice

In [ ]:
editor.set_parameters(VIN=2)
editor.save_netlist(f"{OUTPUT_DIR}/saved.spice")

In [ ]:
print(OUTPUT_DIR)

### saving and reading RAW files

In [ ]:
from typing import List
global_raw_files : List[RawRead] = []
def processing_data(raw_filename, log_filename):
    '''This is a call back function that just prints the filenames'''
    print("Simulation Raw file is %s. The log is %s" % (raw_filename, log_filename))
    # Other code below either using ltsteps.py or raw_read.py
    # log_info = LTSpiceLogReader(log_filename)
    # log_info.read_measures()
    # rise, measures = log_info.dataset["rise_time"]
    global_raw_files.append(RawRead(raw_filename=raw_filename))
    return global_raw_files[raw_filename]

In [ ]:
runtask = runner.run(
    netlist=INITIAL_NETLIST,
    callback=processing_data,
    # run_filename="runfile.spice",
    exe_log=True )
runtask

In [ ]:
runtask.get_results()

In [ ]:
runner.sim_info()

In [ ]:
raw = global_raw_files[0]
raw.get_trace_names()

In [ ]:
raw.get_plot_names()

In [ ]:
raw.get_plot_name()

In [ ]:
for plot in raw.plots:
    print(plot.get_plot_name())
    print(len(plot.get_trace("v(vout)").get_wave()))
    

In [ ]:
raw.get_trace("v(vout)").get_wave()

## Batch Runner

In [ ]:
# TODO

# Optimizer Logic

In [ ]:
# TODO